In [1]:
!pip install torch torchvision

In [2]:
import torch
import torchvision
import torch.nn.functional as F  # Parameterless functions, like (some) activation functions
import torchvision.datasets as datasets  # Standard datasets
import torchvision.transforms as transforms  # Transformations we can perform on our dataset for augmentation
from torch import optim  # For optimizers like SGD, Adam, etc.
import torch.nn as nn  # All neural network modules
from torch.utils.data import DataLoader, random_split
from torchsummary import summary
from tqdm import tqdm  # For nice progress bar!
import matplotlib.pyplot as plt

Setting the device
-

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Pretrained Model
-

mobilenet_v2 has already learned to identify edges, textures, and shapes from millions of images in ImageNet

In [4]:
num_classes = 4 # 4 categories
model = torchvision.models.mobilenet_v2(weights="DEFAULT")
model.avgpool = nn.Identity()
# The last conv layer of MobileNetV2 has 1280 channels.
# For 75x75 input, the spatial map is 3x3.
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2), # MobileNet usually uses dropout before the linear layer
    nn.Linear(1280 , 100), 
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)
model.to(device)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [5]:
model.last_channel

1280

In [6]:
summary(model,(3,24,24))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 12, 12]             864
       BatchNorm2d-2           [-1, 32, 12, 12]              64
             ReLU6-3           [-1, 32, 12, 12]               0
            Conv2d-4           [-1, 32, 12, 12]             288
       BatchNorm2d-5           [-1, 32, 12, 12]              64
             ReLU6-6           [-1, 32, 12, 12]               0
            Conv2d-7           [-1, 16, 12, 12]             512
       BatchNorm2d-8           [-1, 16, 12, 12]              32
  InvertedResidual-9           [-1, 16, 12, 12]               0
           Conv2d-10           [-1, 96, 12, 12]           1,536
      BatchNorm2d-11           [-1, 96, 12, 12]             192
            ReLU6-12           [-1, 96, 12, 12]               0
           Conv2d-13             [-1, 96, 6, 6]             864
      BatchNorm2d-14             [-1, 9

Hyper-Parameters
-

In [7]:
# Updated Hyper-parameters
in_channels =3
learning_rate = 3e-4 #karpathy's constant
num_epochs = 3
batch_size =16

Load the Data
-

Input Size, Conv Base Output Calculation,   Final Feature Map (Channels * H * W)
1. 32,       |      32/32=1,             |           512×1×1=512
2. 75,       |    75/32=2.34→2,          |           512×2×2=2048

In [8]:
trans_train = transforms.Compose([transforms.Resize(150), 
                                  transforms.RandomRotation(30),
                                  transforms.RandomResizedCrop(32),
                                  transforms.RandomHorizontalFlip(),
                                  transforms. ToTensor()])

data_dir = "C:/Users/dbda.STUDENTSDC/Desktop/Practical Machine Learning/SheepFaceImages"
full_dataset = datasets.ImageFolder(data_dir, transform=trans_train)
dataset_size =len(full_dataset)
train_size = int(0.8*dataset_size)
test_size = dataset_size - train_size
train_dataset, test_dataset = random_split(full_dataset,[train_size,test_size])
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle= True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle =False)
type(train_dataset)

torch.utils.data.dataset.Subset

Loss and optimizer
-

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Train Network
--

In [10]:
for epoch in range(num_epochs):
    loop =tqdm(train_loader,leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)
        targets = F.one_hot(targets,num_classes=num_classes).float()
        # forward
        scores = model(data)
        loss = criterion(scores, targets)
        
        # backward
        optimizer.zero_grad()
        loss.backward()

        # gradient descent or adam step
        optimizer.step()

        loop.set_description(f"Epoch: [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

Epoch: [3/3]: 100%|██████████| 84/84 [00:12<00:00,  6.52it/s, loss=1.2]  


##### Check accuracy on training & test to see how good our model

In [11]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)
            # predictions = (scores[:,1]>0.5).long()
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
        
    model.train()
    return float(num_correct) / num_samples


In [12]:
# Check accuracy on both sets
train_acc = check_accuracy(train_loader, model)
test_acc = check_accuracy(test_loader, model)

print(f"Accuracy on training set: {train_acc*100:.2f}%")
print(f"Accuracy on test set: {test_acc*100:.2f}%")

Accuracy on training set: 54.84%
Accuracy on test set: 52.68%
